<a href="https://colab.research.google.com/github/Ash100/beta_tester/blob/main/updated_Basic_of_DFT_calculations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

My name is **Dr. Ashfaq Ahmad**, working in the field of structure biology and bioinformatics. The DFT notebook is designed for teaching and research purposes. If you want to use this pipeline for commercial purpose, please contact. Detailed video tutorial of this notebook can be found on [Bioinformatic Insights](https://youtu.be/qGqXS4aWgAg).

**Announcement**

If you need assistance in developing a pipeline for your research protocol in a paid capacity, feel free to reach out from the channel page or github. I’d be happy to help!

In [ ]:
#@title Install necessary libraries
!pip install pyscf ase matplotlib numpy ipywidgets pubchempy rdkit reportlab

# Density Functional Theory (DFT) Analysis

Density Functional Theory (DFT) is a computational quantum mechanical modeling method used to investigate the electronic structure of many-body systems, especially atoms, molecules, and the condensed phases. DFT is among the most popular and versatile methods available in condensed-matter physics, computational physics, and computational chemistry.

## Basic Concepts of DFT

1. **Hohenberg-Kohn Theorems**:
   - The ground state properties of a many-electron system are uniquely determined by the electron density.
   - The electron density that minimizes the total energy is the true ground state electron density.

2. **Kohn-Sham Equations**:
   - The Kohn-Sham equations are a set of self-consistent equations that allow us to find the electron density of a system.

3. **Exchange-Correlation Functionals**:
   - The exchange-correlation functional accounts for the non-classical part of the electron-electron interaction.

In [ ]:
#@title Calculate HOMO and LUMO Water
for from pyscf import gto, dft, tools

# Define a molecule (e.g., water)
mol = gto.Mole()
mol.atom = '''
O 0 0 0
H 0 1 0
H 0 0 1
'''
mol.basis = 'sto-3g'  # Basis set
mol.build()

# Perform DFT calculation
mf = dft.RKS(mol)
mf.xc = 'b3lyp'  # Use B3LYP functional
mf.kernel()

# Extract HOMO and LUMO energies
mo_energy = mf.mo_energy  # Molecular orbital energies
mo_occ = mf.mo_occ        # Molecular orbital occupancies

# HOMO is the highest occupied molecular orbital
homo_index = int(sum(mo_occ) // 2 - 1)  # For closed-shell systems
homo_energy = mo_energy[homo_index]

# LUMO is the lowest unoccupied molecular orbital
lumo_index = homo_index + 1
lumo_energy = mo_energy[lumo_index]

# Print results
print("Total Energy (B3LYP):", mf.e_tot)
print(f"HOMO Energy: {homo_energy:.4f} Hartree")
print(f"LUMO Energy: {lumo_energy:.4f} Hartree")

# Visualize HOMO and LUMO orbitals
tools.cubegen.orbital(mol, 'homo.cube', mf.mo_coeff[:, homo_index])
tools.cubegen.orbital(mol, 'lumo.cube', mf.mo_coeff[:, lumo_index])
print("HOMO orbital saved to 'homo.cube'")
print("LUMO orbital saved to 'lumo.cube'")

In [ ]:
#@title Visualize the HOMO and LUMO
Import necessary modules
import matplotlib.pyplot as plt
import numpy as np
from pyscf import gto, dft, tools
from ase.io.cube import read_cube_data

# Define a molecule (e.g., water)
mol = gto.Mole()
mol.atom = '''
O 0 0 0
H 0 1 0
H 0 0 1
'''
mol.basis = 'sto-3g'  # Basis set
mol.build()

# Perform DFT calculation
mf = dft.RKS(mol)
mf.xc = 'b3lyp'  # Use B3LYP functional
mf.kernel()

# Extract HOMO and LUMO indices
mo_occ = mf.mo_occ  # Molecular orbital occupancies
homo_index = int(sum(mo_occ) // 2 - 1)  # Index of the HOMO
lumo_index = homo_index + 1  # Index of the LUMO

# Generate cube files for HOMO and LUMO
tools.cubegen.orbital(mol, 'homo.cube', mf.mo_coeff[:, homo_index])
tools.cubegen.orbital(mol, 'lumo.cube', mf.mo_coeff[:, lumo_index])

# Function to visualize a cube file
def visualize_cube(cube_file, title):
    # Read the cube file
    data, atoms = read_cube_data(cube_file)

    # Extract a 2D slice (middle plane along the z-axis)
    slice_index = data.shape[2] // 2
    slice_data = data[:, :, slice_index]

    # Plot the 2D slice
    plt.imshow(slice_data, cmap='RdBu', origin='lower')
    plt.colorbar(label='Electron Density')
    plt.title(title)
    plt.show()

# Visualize HOMO
visualize_cube('homo.cube', 'HOMO of Water Molecule')

# Visualize LUMO
visualize_cube('lumo.cube', 'LUMO of Water Molecule')

In [ ]:
#@title Calculate Molecule of your choice with **Desired model** from PubChem
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem
from pyscf import gto, dft
from pyscf.tools import cubegen
import matplotlib.pyplot as plt
from ase.io.cube import read_cube_data
import ipywidgets as widgets
from IPython.display import display

# Create a text box and button for user input
cid_input = widgets.Text(value='', placeholder='Enter PubChem CID (e.g., 2519 for caffeine)', description='CID:')
calculate_button = widgets.Button(description="Calculate HOMO and LUMO")

# Output area to display results
output = widgets.Output()

# Function to handle button click
def on_calculate_button_click(b):
    with output:
        output.clear_output()  # Clear previous output
        try:
            cid = int(cid_input.value)
            print(f"Fetching molecule with CID: {cid}")

            # Fetch the molecule and generate 3D coordinates
            atoms, coords = fetch_and_optimize_molecule(cid)

            # Perform DFT calculation and visualize the HOMO and LUMO
            calculate_and_visualize_homo_lumo(atoms, coords)
        except Exception as e:
            print(f"An error occurred: {e}")

# Attach the function to the button
calculate_button.on_click(on_calculate_button_click)

# Display the input box, button, and output area
display(cid_input, calculate_button, output)

# Function to fetch molecule from PubChem and generate 3D coordinates
def fetch_and_optimize_molecule(cid):
    # Fetch the molecule from PubChem
    compound = pcp.Compound.from_cid(cid)
    smiles = compound.canonical_smiles
    print(f"Molecule Name: {compound.iupac_name}")
    print(f"Molecular Formula: {compound.molecular_formula}")
    print(f"SMILES: {smiles}")

    # Convert SMILES to 3D structure using RDKit
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)  # Add hydrogens
    AllChem.EmbedMolecule(mol)  # Generate 3D coordinates
    AllChem.MMFFOptimizeMolecule(mol)  # Optimize geometry

    # Extract atomic coordinates
    conf = mol.GetConformer()
    atoms = [mol.GetAtomWithIdx(i).GetSymbol() for i in range(mol.GetNumAtoms())]
    coords = [conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())]

    return atoms, coords

# Function to perform DFT calculation and visualize HOMO and LUMO
def calculate_and_visualize_homo_lumo(atoms, coords):
    # Define the molecule in PySCF
    mol_pyscf = gto.Mole()
    mol_pyscf.atom = [[atom, (coord.x, coord.y, coord.z)] for atom, coord in zip(atoms, coords)]
    mol_pyscf.basis = 'sto-3g'  #@param ["sto-3g", "3-21g", "6-31g", "6-31g*", "6-31g**", "6-311g", "6-311g*", "6-311g**", "cc-pVDZ", "cc-pVTZ", "cc-pVQZ", "cc-pV5Z", "6-31+g", "6-31++g", "6-311+g", "6-311++g", "aug-cc-pVDZ", "aug-cc-pVTZ", "aug-cc-pVQZ", "aug-cc-pV5Z", "lanl2dz", "def2-SVP"]
    mol_pyscf.build()

    # Perform DFT calculation
    mf = dft.RKS(mol_pyscf)
    mf.xc = 'b3lyp'  # Use B3LYP functional
    mf.kernel()

    # Print the total energy
    print(f"Total Energy (B3LYP): {mf.e_tot}")

    # Extract HOMO and LUMO indices
    mo_occ = mf.mo_occ  # Molecular orbital occupancies
    homo_index = int(sum(mo_occ) // 2 - 1)  # Index of the HOMO
    lumo_index = homo_index + 1  # Index of the LUMO

    # Generate cube files for HOMO and LUMO
    cubegen.orbital(mol_pyscf, 'homo.cube', mf.mo_coeff[:, homo_index])
    cubegen.orbital(mol_pyscf, 'lumo.cube', mf.mo_coeff[:, lumo_index])

    # Function to visualize a cube file
    def visualize_cube(cube_file, title):
        # Read the cube file
        data, atoms = read_cube_data(cube_file)

        # Extract a 2D slice (middle plane along the z-axis)
        slice_index = data.shape[2] // 2
        slice_data = data[:, :, slice_index]

        # Plot the 2D slice
        plt.imshow(slice_data, cmap='RdBu', origin='lower')
        plt.colorbar(label='Electron Density')
        plt.title(title)
        plt.show()

    # Visualize HOMO
    visualize_cube('homo.cube', 'HOMO of the Molecule')

    # Visualize LUMO
    visualize_cube('lumo.cube', 'LUMO of the Molecule')

    # Print HOMO and LUMO energies
    homo_energy = mf.mo_energy[homo_index]
    lumo_energy = mf.mo_energy[lumo_index]
    print(f"HOMO Energy: {homo_energy:.4f} Hartree")
    print(f"LUMO Energy: {lumo_energy:.4f} Hartree")

In [ ]:
#@title Calculate HOMO-LUMO gap for molecule of your choice with **Desired model** from PubChem
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem
from pyscf import gto, dft
from pyscf.tools import cubegen
import matplotlib.pyplot as plt
from ase.io.cube import read_cube_data
import ipywidgets as widgets
from IPython.display import display

# Create a text box and button for user input
cid_input = widgets.Text(value='', placeholder='Enter PubChem CID (e.g., 2519 for caffeine)', description='CID:')
calculate_button = widgets.Button(description="Calculate HOMO and LUMO")

# Output area to display results
output = widgets.Output()

# Function to handle button click
def on_calculate_button_click(b):
    with output:
        output.clear_output()  # Clear previous output
        try:
            cid = int(cid_input.value)
            print(f"Fetching molecule with CID: {cid}")

            # Fetch the molecule and generate 3D coordinates
            atoms, coords = fetch_and_optimize_molecule(cid)

            # Perform DFT calculation and visualize the HOMO and LUMO
            calculate_and_visualize_homo_lumo(atoms, coords)
        except Exception as e:
            print(f"An error occurred: {e}")

# Attach the function to the button
calculate_button.on_click(on_calculate_button_click)

# Display the input box, button, and output area
display(cid_input, calculate_button, output)

# Function to fetch molecule from PubChem and generate 3D coordinates
def fetch_and_optimize_molecule(cid):
    # Fetch the molecule from PubChem
    compound = pcp.Compound.from_cid(cid)
    smiles = compound.canonical_smiles
    print(f"Molecule Name: {compound.iupac_name}")
    print(f"Molecular Formula: {compound.molecular_formula}")
    print(f"SMILES: {smiles}")

    # Convert SMILES to 3D structure using RDKit
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)  # Add hydrogens
    AllChem.EmbedMolecule(mol)  # Generate 3D coordinates
    AllChem.MMFFOptimizeMolecule(mol)  # Optimize geometry

    # Extract atomic coordinates
    conf = mol.GetConformer()
    atoms = [mol.GetAtomWithIdx(i).GetSymbol() for i in range(mol.GetNumAtoms())]
    coords = [conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())]

    return atoms, coords

# Function to perform DFT calculation and visualize HOMO and LUMO
def calculate_and_visualize_homo_lumo(atoms, coords):
    # Define the molecule in PySCF
    mol_pyscf = gto.Mole()
    mol_pyscf.atom = [[atom, (coord.x, coord.y, coord.z)] for atom, coord in zip(atoms, coords)]
    mol_pyscf.basis = 'sto-3g'  #@param ["sto-3g", "3-21g", "6-31g", "6-31g*", "6-31g**", "6-311g", "6-311g*", "6-311g**", "cc-pVDZ", "cc-pVTZ", "cc-pVQZ", "cc-pV5Z", "6-31+g", "6-31++g", "6-311+g", "6-311++g", "aug-cc-pVDZ", "aug-cc-pVTZ", "aug-cc-pVQZ", "aug-cc-pV5Z", "lanl2dz", "def2-SVP"]
    mol_pyscf.build()

    # Perform DFT calculation
    mf = dft.RKS(mol_pyscf)
    mf.xc = 'b3lyp'  # Use B3LYP functional
    mf.kernel()

    # Print the total energy
    print(f"Total Energy (B3LYP): {mf.e_tot}")

    # Extract HOMO and LUMO indices
    mo_occ = mf.mo_occ  # Molecular orbital occupancies
    homo_index = int(sum(mo_occ) // 2 - 1)  # Index of the HOMO
    lumo_index = homo_index + 1  # Index of the LUMO

    # Generate cube files for HOMO and LUMO
    cubegen.orbital(mol_pyscf, 'homo.cube', mf.mo_coeff[:, homo_index])
    cubegen.orbital(mol_pyscf, 'lumo.cube', mf.mo_coeff[:, lumo_index])

    # Function to visualize a cube file
    def visualize_cube(cube_file, title):
        # Read the cube file
        data, atoms = read_cube_data(cube_file)

        # Extract a 2D slice (middle plane along the z-axis)
        slice_index = data.shape[2] // 2
        slice_data = data[:, :, slice_index]

        # Plot the 2D slice
        plt.imshow(slice_data, cmap='RdBu', origin='lower')
        plt.colorbar(label='Electron Density')
        plt.title(title)
        plt.show()

    # Visualize HOMO
    visualize_cube('homo.cube', 'HOMO of the Molecule')

    # Visualize LUMO
    visualize_cube('lumo.cube', 'LUMO of the Molecule')

    # Print HOMO and LUMO energies
    homo_energy = mf.mo_energy[homo_index]
    lumo_energy = mf.mo_energy[lumo_index]
    print(f"HOMO Energy: {homo_energy:.4f} Hartree")
    print(f"LUMO Energy: {lumo_energy:.4f} Hartree")

    # Calculate and print the HOMO-LUMO gap
    homo_lumo_gap = lumo_energy - homo_energy
    print(f"HOMO-LUMO Gap: {homo_lumo_gap:.4f} Hartree")
    print(f"HOMO-LUMO Gap: {homo_lumo_gap * 27.2114:.4f} eV")  # Convert to electron volts (eV)

**Warning**: Some of the codes generates a report in PDF. However, this code is in testing phase, and I personally did not test it on different compounds, therefore, I strongly suggest to please consult an expert, in case you want to publish it.

## ONLY USE THE BELOW CODE FOr YOUR OWN NOVEL COMPOUNDS - **Not available in Databases**

In [ ]:
#@title Test your synthesized molecule
from rdkit import Chem
from rdkit.Chem import AllChem
from pyscf import gto, dft
from pyscf.tools import cubegen
import matplotlib.pyplot as plt
from ase.io.cube import read_cube_data
import ipywidgets as widgets
from IPython.display import display
import numpy as np
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from mpl_toolkits.mplot3d import Axes3D
from google.colab import files

# Create a button for uploading SDF file
upload_button = widgets.FileUpload(description="Upload SDF File", accept='.sdf', multiple=False)
calculate_button = widgets.Button(description="Generate HOMO and LUMO Report")

# Output area to display results
output = widgets.Output()

# Function to handle button click
def on_calculate_button_click(b):
    with output:
        output.clear_output()  # Clear previous output
        try:
            # Get the uploaded SDF file
            uploaded_file = list(upload_button.value.values())[0]
            sdf_content = uploaded_file['content'].decode('utf-8')

            # Read the molecule from the SDF file
            mol = Chem.MolFromMolBlock(sdf_content)
            if mol is None:
                raise ValueError("Invalid SDF file. Please upload a valid SDF file.")

            # Generate 3D coordinates
            atoms, coords = generate_3d_coordinates(mol)

            # Perform DFT calculation and analyze HOMO and LUMO
            report, homo_data, lumo_data, homo_atoms = calculate_and_analyze_homo_lumo(atoms, coords)

            # Display the report
            print("\n--- HOMO and LUMO Analysis Report ---")
            print(report)

            # Plot the HOMO and LUMO
            plot_orbitals(homo_data, lumo_data, homo_atoms)

            # Save the report as a PDF
            save_report_as_pdf(report, homo_data, lumo_data, homo_atoms)
            print("\nReport saved as 'homo_lumo_report.pdf'.")
        except Exception as e:
            print(f"An error occurred: {e}")

# Attach the function to the button
calculate_button.on_click(on_calculate_button_click)

# Display the upload button, calculate button, and output area
display(upload_button, calculate_button, output)

# Function to generate 3D coordinates from an RDKit molecule
def generate_3d_coordinates(mol):
    # Add hydrogens and generate 3D coordinates
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)  # Generate 3D coordinates
    AllChem.MMFFOptimizeMolecule(mol)  # Optimize geometry

    # Extract atomic coordinates
    conf = mol.GetConformer()
    atoms = [mol.GetAtomWithIdx(i).GetSymbol() for i in range(mol.GetNumAtoms())]
    coords = [conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())]

    return atoms, coords

# Function to perform DFT calculation and analyze HOMO and LUMO
def calculate_and_analyze_homo_lumo(atoms, coords):
    # Define the molecule in PySCF
    mol_pyscf = gto.Mole()
    mol_pyscf.atom = [[atom, (coord.x, coord.y, coord.z)] for atom, coord in zip(atoms, coords)]
    mol_pyscf.basis = 'sto-3g'  #@param ["sto-3g", "3-21g", "6-31g", "6-31g*", "6-31g**", "6-311g", "6-311g*", "6-311g**", "cc-pVDZ", "cc-pVTZ", "cc-pVQZ", "cc-pV5Z", "6-31+g", "6-31++g", "6-311+g", "6-311++g", "aug-cc-pVDZ", "aug-cc-pVTZ", "aug-cc-pVQZ", "aug-cc-pV5Z", "lanl2dz", "def2-SVP"]
    mol_pyscf.build()

    # Perform DFT calculation
    mf = dft.RKS(mol_pyscf)
    mf.xc = 'b3lyp'  # Use B3LYP functional
    mf.kernel()

    # Generate cube files for HOMO and LUMO
    homo_index = mol_pyscf.nelectron // 2 - 1  # Index of the HOMO
    lumo_index = homo_index + 1  # Index of the LUMO
    cubegen.orbital(mol_pyscf, 'homo.cube', mf.mo_coeff[:, homo_index])
    cubegen.orbital(mol_pyscf, 'lumo.cube', mf.mo_coeff[:, lumo_index])

    # Load and analyze the HOMO and LUMO cube files
    homo_data, homo_atoms = read_cube_data('homo.cube')
    lumo_data, _ = read_cube_data('lumo.cube')

    # Analyze the HOMO and LUMO data
    report = analyze_orbitals(homo_data, lumo_data, homo_atoms)

    # Add total energy to the report
    report += f"\nTotal Energy (B3LYP): {mf.e_tot:.6f} Hartree"

    return report, homo_data, lumo_data, homo_atoms

# Function to analyze HOMO and LUMO data and generate a report
def analyze_orbitals(homo_data, lumo_data, homo_atoms):
    # Analyze HOMO
    homo_total_density = np.sum(homo_data)
    homo_max_density = np.max(homo_data)
    homo_min_density = np.min(homo_data)
    homo_symmetry = "symmetric" if np.allclose(homo_data, np.flip(homo_data)) else "asymmetric"
    homo_localization = "delocalized" if np.std(homo_data) < 0.5 else "localized"

    # Analyze LUMO
    lumo_total_density = np.sum(lumo_data)
    lumo_max_density = np.max(lumo_data)
    lumo_min_density = np.min(lumo_data)
    lumo_symmetry = "symmetric" if np.allclose(lumo_data, np.flip(lumo_data)) else "asymmetric"
    lumo_localization = "delocalized" if np.std(lumo_data) < 0.5 else "localized"

    # Generate the report
    report = f"""
--- HOMO and LUMO Analysis Report ---
1. HOMO Analysis:
   - Total Electron Density: {homo_total_density:.6f}
   - Maximum Electron Density: {homo_max_density:.6f}
   - Minimum Electron Density: {homo_min_density:.6f}
   - Symmetry: The HOMO is {homo_symmetry}.
   - Localization: The HOMO is {homo_localization}.

2. LUMO Analysis:
   - Total Electron Density: {lumo_total_density:.6f}
   - Maximum Electron Density: {lumo_max_density:.6f}
   - Minimum Electron Density: {lumo_min_density:.6f}
   - Symmetry: The LUMO is {lumo_symmetry}.
   - Localization: The LUMO is {lumo_localization}.

3. Reactivity Insights:
   - The HOMO represents the most loosely bound electrons, which are likely to participate in chemical reactions.
   - The LUMO represents the lowest energy unoccupied orbital, which can accept electrons in reactions.
"""

    return report

# Function to plot the HOMO and LUMO
def plot_orbitals(homo_data, lumo_data, homo_atoms):
    # Plot HOMO
    plot_orbital(homo_data, "HOMO of the Molecule (2D Slice)", "homo_2d_slice.png")
    plot_3d_isosurface(homo_data, "HOMO of the Molecule (3D Isosurface)", "homo_3d_isosurface.png")

    # Plot LUMO
    plot_orbital(lumo_data, "LUMO of the Molecule (2D Slice)", "lumo_2d_slice.png")
    plot_3d_isosurface(lumo_data, "LUMO of the Molecule (3D Isosurface)", "lumo_3d_isosurface.png")

# Function to plot a 2D slice of an orbital
def plot_orbital(data, title, filename):
    plt.figure(figsize=(8, 6))
    plt.imshow(data[:, :, data.shape[2] // 2], cmap='RdBu', origin='lower')
    plt.colorbar(label='Electron Density')
    plt.title(title)
    plt.savefig(filename)  # Save the 2D plot
    plt.show()

# Function to plot a 3D isosurface of an orbital
def plot_3d_isosurface(data, title, filename):
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    x, y, z = np.mgrid[:data.shape[0], :data.shape[1], :data.shape[2]]
    norm = plt.Normalize(vmin=data.min(), vmax=data.max())
    scatter = ax.scatter(x, y, z, c=data.flatten(), cmap='RdBu', norm=norm, alpha=0.3)
    fig.colorbar(scatter, label='Electron Density')
    ax.set_title(title)
    plt.savefig(filename)  # Save the 3D plot
    plt.show()

# Function to save the report as a PDF
def save_report_as_pdf(report, homo_data, lumo_data, homo_atoms):
    # Create a PDF file
    pdf = canvas.Canvas("homo_lumo_report.pdf", pagesize=letter)
    pdf.setFont("Helvetica", 12)

    # Add the report text
    pdf.drawString(50, 750, "HOMO and LUMO Analysis Report")
    y = 730
    for line in report.split('\n'):
        pdf.drawString(50, y, line)
        y -= 15

    # Add the HOMO 2D plot
    pdf.drawString(50, y - 20, "2D Slice of the HOMO:")
    pdf.drawImage('homo_2d_slice.png', 50, y - 220, width=400, height=200)

    # Add the HOMO 3D plot
    pdf.drawString(50, y - 240, "3D Isosurface of the HOMO:")
    pdf.drawImage('homo_3d_isosurface.png', 50, y - 440, width=400, height=200)

    # Add the LUMO 2D plot
    pdf.drawString(50, y - 460, "2D Slice of the LUMO:")
    pdf.drawImage('lumo_2d_slice.png', 50, y - 660, width=400, height=200)

    # Add the LUMO 3D plot
    pdf.drawString(50, y - 680, "3D Isosurface of the LUMO:")
    pdf.drawImage('lumo_3d_isosurface.png', 50, y - 880, width=400, height=200)

    # Save the PDF
    pdf.save()

In [6]:
#@title Calculate HOMO-LUMO Gap for Novel Compounds
from rdkit import Chem
from rdkit.Chem import AllChem
from pyscf import gto, dft
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

# Create a button for uploading SDF file
upload_button = widgets.FileUpload(description="Upload SDF File", accept='.sdf', multiple=False)
calculate_button = widgets.Button(description="Calculate HOMO-LUMO Gap")

# Output area to display results
output = widgets.Output()

# Function to handle button click
def on_calculate_button_click(b):
    with output:
        output.clear_output()  # Clear previous output
        try:
            # Get the uploaded SDF file
            uploaded_file = list(upload_button.value.values())[0]
            sdf_content = uploaded_file['content'].decode('utf-8')

            # Read the molecule from the SDF file
            mol = Chem.MolFromMolBlock(sdf_content)
            if mol is None:
                raise ValueError("Invalid SDF file. Please upload a valid SDF file.")

            # Generate 3D coordinates
            atoms, coords = generate_3d_coordinates(mol)

            # Perform DFT calculation and extract HOMO and LUMO energies
            homo_energy, lumo_energy = calculate_homo_lumo_gap(atoms, coords)

            # Calculate the HOMO-LUMO gap
            homo_lumo_gap = lumo_energy - homo_energy

            # Display the results
            print("\n--- HOMO-LUMO Gap Analysis ---")
            print(f"HOMO Energy: {homo_energy:.6f} Hartree")
            print(f"LUMO Energy: {lumo_energy:.6f} Hartree")
            print(f"HOMO-LUMO Gap: {homo_lumo_gap:.6f} Hartree")
        except Exception as e:
            print(f"An error occurred: {e}")

# Attach the function to the button
calculate_button.on_click(on_calculate_button_click)

# Display the upload button, calculate button, and output area
display(upload_button, calculate_button, output)

# Function to generate 3D coordinates from an RDKit molecule
def generate_3d_coordinates(mol):
    # Add hydrogens and generate 3D coordinates
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)  # Generate 3D coordinates
    AllChem.MMFFOptimizeMolecule(mol)  # Optimize geometry

    # Extract atomic coordinates
    conf = mol.GetConformer()
    atoms = [mol.GetAtomWithIdx(i).GetSymbol() for i in range(mol.GetNumAtoms())]
    coords = [conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())]

    return atoms, coords

# Function to perform DFT calculation and extract HOMO and LUMO energies
def calculate_homo_lumo_gap(atoms, coords):
    # Define the molecule in PySCF
    mol_pyscf = gto.Mole()
    mol_pyscf.atom = [[atom, (coord.x, coord.y, coord.z)] for atom, coord in zip(atoms, coords)]
    mol_pyscf.basis = 'sto-3g'  #@param ["sto-3g", "3-21g", "6-31g", "6-31g*", "6-31g**", "6-311g", "6-311g*", "6-311g**", "cc-pVDZ", "cc-pVTZ", "cc-pVQZ", "cc-pV5Z", "6-31+g", "6-31++g", "6-311+g", "6-311++g", "aug-cc-pVDZ", "aug-cc-pVTZ", "aug-cc-pVQZ", "aug-cc-pV5Z", "lanl2dz", "def2-SVP"]
    mol_pyscf.build()

    # Perform DFT calculation
    mf = dft.RKS(mol_pyscf)
    mf.xc = 'b3lyp'  # Use B3LYP functional
    mf.kernel()

    # Extract HOMO and LUMO energies
    mo_energy = mf.mo_energy
    homo_index = mol_pyscf.nelectron // 2 - 1  # Index of the HOMO
    lumo_index = homo_index + 1  # Index of the LUMO
    homo_energy = mo_energy[homo_index]
    lumo_energy = mo_energy[lumo_index]

    return homo_energy, lumo_energy

FileUpload(value={}, accept='.sdf', description='Upload SDF File')

Button(description='Calculate HOMO-LUMO Gap', style=ButtonStyle())

Output()

## Understanding HOMO and LUMO

### What is HOMO?
- **HOMO** stands for **Highest Occupied Molecular Orbital**.
- It is the highest energy orbital that contains electrons in a molecule.
- Electrons in the HOMO are the most loosely bound and are often involved in chemical reactions, such as donating electrons during bond formation.
- The energy of the HOMO is a measure of how easily a molecule can donate electrons (its **ionization potential**).

### What is LUMO?
- **LUMO** stands for **Lowest Unoccupied Molecular Orbital**.
- It is the lowest energy orbital that does not contain electrons in a molecule.
- The LUMO is where electrons can be accepted during chemical reactions, such as bond formation or reduction reactions.
- The energy of the LUMO is a measure of how easily a molecule can accept electrons (its **electron affinity**).

---

### **Key Differences Between HOMO and LUMO**

| Feature                | HOMO (Highest Occupied Molecular Orbital)       | LUMO (Lowest Unoccupied Molecular Orbital)      |
|------------------------|------------------------------------------------|------------------------------------------------|
| **Definition**         | Highest energy orbital with electrons          | Lowest energy orbital without electrons        |
| **Electron Status**    | Occupied by electrons                          | Unoccupied (empty)                             |
| **Role in Reactions**  | Donates electrons (oxidation)                  | Accepts electrons (reduction)                  |
| **Energy Level**       | Higher energy than LUMO                        | Lower energy than HOMO                         |
| **Chemical Reactivity**| Determines the molecule's ability to donate electrons | Determines the molecule's ability to accept electrons |

---

### **Why Are HOMO and LUMO Important?**
1. **Chemical Reactivity**:
   - The energy gap between the HOMO and LUMO (called the **HOMO-LUMO gap**) determines the molecule's reactivity.
   - A smaller HOMO-LUMO gap generally means the molecule is more reactive.

2. **Optical Properties**:
   - The HOMO-LUMO gap is related to the molecule's absorption and emission of light.
   - For example, in organic semiconductors, the HOMO-LUMO gap determines the material's bandgap and its ability to conduct electricity.

3. **Redox Reactions**:
   - The HOMO is involved in oxidation reactions (losing electrons).
   - The LUMO is involved in reduction reactions (gaining electrons).

4. **Drug Design**:
   - In medicinal chemistry, the HOMO and LUMO energies of drug molecules can influence their binding affinity to biological targets.

---

### **Visualizing HOMO and LUMO**
- **HOMO**: Represents the distribution of the most loosely bound electrons in the molecule.
- **LUMO**: Represents the distribution of the lowest energy orbitals available to accept electrons.

For example:
- In a **2D slice**, the HOMO and LUMO orbitals are visualized as electron density maps.
- In a **3D isosurface**, the HOMO and LUMO orbitals are represented as surfaces where the electron density is constant.

---

### **Example: HOMO and LUMO in Action**
Consider a simple molecule like **ethylene (C₂H₄)**:
- The **HOMO** is the π-bonding orbital, which contains the electrons involved in the double bond.
- The **LUMO** is the π* (pi-star) anti-bonding orbital, which is empty and can accept electrons.

When ethylene reacts with another molecule:
- Electrons from the HOMO of ethylene can be donated to form a new bond.
- Electrons can be accepted into the LUMO of ethylene to break the double bond.

---

### **Conclusion**
Understanding the HOMO and LUMO of a molecule provides insights into its:
- **Reactivity**: How easily it can donate or accept electrons.
- **Optical Properties**: How it interacts with light.
- **Chemical Behavior**: Its role in redox reactions and bond formation.

By analyzing the HOMO and LUMO, chemists can predict and design molecules with specific properties for applications in materials science, drug design, and catalysis.

## Analysis and Discussion of Results

Let's discuss these results, such as the total energy, molecular orbitals, and electron density

- **Total Energy**: The total energy of the system is calculated using the B3LYP functional.
- **Molecular Orbitals**: The HOMO (Highest Occupied Molecular Orbital) is visualized, showing the electron density distribution.
- **Electron Density**: The electron density can be analyzed to understand the chemical bonding and reactivity of the molecule.

# Interpreting the HOMO (Highest Occupied Molecular Orbital) Figure

The **HOMO (Highest Occupied Molecular Orbital)** figure visualizes the electron density distribution in the highest energy orbital that is occupied by electrons in a molecule. This orbital is significant because it plays a key role in chemical reactions, particularly in processes like **electron donation** (e.g., in redox reactions) and **bond formation**.

---

## What Does the HOMO Represent?
- The HOMO represents the **most loosely bound electrons** in the molecule.
- These electrons are the most likely to participate in chemical reactions, such as forming bonds with other molecules or donating electrons.
- The shape and distribution of the HOMO provide insights into the **reactivity** and **electronic properties** of the molecule.

---

## Key Features to Look For in the HOMO Figure

### 1. **Electron Density Distribution**
- The **bright regions** (often colored red or blue in the plot) represent areas of high electron density.
- The **dim regions** (often white or transparent) represent areas of low or no electron density.
- The electron density is typically concentrated around **atoms** or **bonds** where the electrons are most likely to be found.

### 2. **Symmetry and Shape**
- The HOMO often reflects the **symmetry** of the molecule. For example, in a symmetric molecule like benzene, the HOMO will have a symmetric distribution of electron density.
- The shape of the HOMO can be **bonding**, **antibonding**, or **non-bonding**:
  - **Bonding orbitals**: Electron density is concentrated between atoms, indicating stabilizing interactions.
  - **Antibonding orbitals**: Electron density is concentrated outside the bond region, indicating destabilizing interactions.
  - **Non-bonding orbitals**: Electron density is localized on a single atom (e.g., lone pairs).

### 3. **Localization vs. Delocalization**
- If the electron density is **localized** on a specific atom or bond, it suggests that the HOMO is associated with a particular functional group or region of the molecule.
- If the electron density is **delocalized** over multiple atoms or bonds, it indicates that the electrons are shared across a larger region of the molecule (e.g., in conjugated systems like benzene or polyenes).

---

## Interpreting the HOMO for Chemical Reactivity
- **Nucleophilic Sites**: Regions with high electron density in the HOMO are potential sites for **nucleophilic attack** (donating electrons).
- **Electron Donation**: The HOMO is the orbital from which electrons are most easily donated in a reaction.
- **Reactivity Trends**: Molecules with **higher energy HOMOs** are generally more reactive because their electrons are less tightly bound.

---

## Example: HOMO of Water (H₂O)
Let’s take the example of water (H₂O), which we used in the DFT calculation earlier.

### **HOMO Characteristics for Water**:
- The HOMO of water is primarily localized on the **oxygen atom**, representing the **lone pairs** of electrons.
- The electron density is concentrated around the oxygen, with little to no density on the hydrogen atoms.
- This indicates that the oxygen atom is the most likely site for **electron donation** (e.g., in hydrogen bonding or nucleophilic reactions).

### **Visualization**:
- In the HOMO plot, you would see bright regions (high electron density) around the oxygen atom and dim regions around the hydrogens.

---

## Example: HOMO of Benzene (C₆H₆)
For a more complex molecule like benzene:

### **HOMO Characteristics for Benzene**:
- The HOMO of benzene is **delocalized** over the entire ring, reflecting the **π-electron system**.
- The electron density is evenly distributed above and below the plane of the ring.
- This delocalization indicates that benzene is highly stable and less reactive compared to molecules with localized HOMOs.

### **Visualization**:
- In the HOMO plot, you would see a symmetric, donut-shaped electron density distribution above and below the ring.

---

## How to Use the HOMO in Analysis
- **Predict Reactivity**: Identify regions of high electron density to predict where the molecule is likely to react.
- **Compare Molecules**: Compare HOMOs of different molecules to understand their relative reactivity or stability.
- **Understand Bonding**: Use the HOMO to analyze bonding patterns, such as conjugation or lone pairs.

---

## Limitations of the HOMO
- The HOMO only provides information about the **highest occupied orbital**. For a complete picture of reactivity, you should also consider the **LUMO (Lowest Unoccupied Molecular Orbital)**, which represents the lowest energy orbital that can accept electrons.
- The HOMO does not account for **dynamic effects** (e.g., solvent interactions or temperature), which can also influence reactivity.

---

## Summary
- The HOMO figure shows the distribution of the most loosely bound electrons in a molecule.
- High electron density regions indicate potential sites for reactivity (e.g., nucleophilic attack).
- The shape and symmetry of the HOMO reflect the molecule’s electronic structure and bonding.
- By analyzing the HOMO, you can gain insights into the molecule’s chemical behavior and reactivity.

## Conclusion

In this tutorial, we have introduced the basic concepts of Density Functional Theory (DFT) and demonstrated how to perform DFT calculations on a simple molecule using `PySCF`. We also visualized the molecular orbitals and analyzed the results.

## DFT Calculations for Vibrational Frequencies and Thermodynamics Properties

### Further Reading
- [PySCF Documentation](https://pyscf.org/)
- [Density Functional Theory: A Practical Introduction by David Sholl and Janice A. Steckel](https://onlinelibrary.wiley.com/doi/book/10.1002/9780470447710)